# OP2 Reader — Test Notebook

In [ ]:
# Setup — reload module and open file
import sys
for mod in list(sys.modules.keys()):
    if "op2_native" in mod:
        del sys.modules[mod]

sys.path.insert(0, r"c:\Users\evanc\Dropbox\Coding\Python\OP2_reader")
from op2_native import OP2

OP2_FILE = r"c:\Users\evanc\Dropbox\Coding\Python\OP2_reader\testing-015.op2"
op2 = OP2(OP2_FILE)
print("Loaded:", OP2_FILE)


In [ ]:
## 1 — File Summary
# All records scanned from the binary file.

In [ ]:
summary = op2.summary()
print(f"{len(summary)} records total")
summary


In [ ]:
## 2 — Displacements (OUGV1)
# Columns: GRID, CP (output coord system), DX, DY, DZ (translations), RX, RY, RZ (rotations)

In [ ]:
displ = op2.displacements()
for sc, df in displ.items():
    print(f"Subcase {sc}: {len(df)} rows")
    display(df.head(10))


In [ ]:
## 3 — Element Stresses (OES1X1 shell)
# Centroid-only:  EID, FD1, SX1, SY1, TXY1, ANG1, MAJOR1, MINOR1, VM1,
#                     FD2, SX2, SY2, TXY2, ANG2, MAJOR2, MINOR2, VM2
# With corners:   EID, GRID (0=centroid), FD1..VM2  →  5 rows/element
# VM1/VM2 = Von Mises (or Max Shear, depending on Nastran STRESS case control)

In [ ]:
stresses = op2.stresses()
for sc, df in stresses.items():
    print(f"Subcase {sc}: {len(df)} rows")
    display(df)


In [ ]:
## 4 — Element Forces (OEF1)
# CQUAD4/CTRIA3: EID, NX, NY, NXY (membrane), MX, MY, MXY (bending), QX, QY (transverse shear)
# CBAR/CBEAM:   EID, BM1A, BM2A, BM1B, BM2B, TS1, TS2, AF, TRQ

In [ ]:
eforces = op2.element_forces()
for sc, df in eforces.items():
    print(f"Subcase {sc}: {len(df)} rows")
    display(df.head(10))


In [ ]:
## 5 — SPC Constraint Forces (OQG1)

In [ ]:
spc = op2.spc_forces()
for sc, df in spc.items():
    nonzero = df[(df[["FX","FY","FZ","MX","MY","MZ"]].abs() > 0).any(axis=1)]
    print(f"Subcase {sc}: {len(df)} rows total, {len(nonzero)} nonzero")
    display(nonzero.head(20))


In [ ]:
## 6 — Applied Loads (OPG1)

In [ ]:
try:
    loads = op2.applied_loads()
    if loads:
        for sc, df in loads.items():
            print(f"Subcase {sc}: {len(df)} rows")
            display(df.head(10))
    else:
        print("No OPG1 tables found in this file.")
except Exception as e:
    print(f"Error: {e}")


In [ ]:
## 7 — Strains (OSTR1)

In [ ]:
try:
    strains = op2.strains()
    if strains:
        for sc, df in strains.items():
            print(f"Subcase {sc}: {len(df)} rows")
            display(df.head(10))
    else:
        print("No OSTR1 tables found in this file.")
except Exception as e:
    print(f"Error: {e}")


In [ ]:
## 8 — Solid / Bar Stresses
# Returns empty for this file (no solid/bar elements), but confirms the API works.

In [ ]:
solid = op2.solid_stresses()
bars  = op2.bar_stresses()
print(f"solid_stresses subcases : {sorted(solid.keys())}")
print(f"bar_stresses   subcases : {sorted(bars.keys())}")
if not solid and not bars:
    print("(none — no solid/bar elements in this file)")


In [ ]:
## 8b — Eigenvalues (LAMA)
# Available for modal (SOL 103) and buckling (SOL 105) analyses.
# Columns: MODE, ORDER, EIGENVALUE (ω², rad²/s²), RADIANS (ω, rad/s),
#          CYCLES (f, Hz), GENM (generalised mass), GENSTIF (generalised stiffness)
eigs = op2.eigenvalues()
if eigs:
    for sc, df in eigs.items():
        print(f"Subcase {sc}: {len(df)} modes")
        display(df)
else:
    print("No LAMA table found (this is a static analysis file).")

In [ ]:
## 9 — Summary Table
import pandas as pd

all_results = {
    "displacements":      op2.displacements(),
    "stresses":           op2.stresses(),
    "stresses_corners":   op2.stresses_with_corners(),
    "solid_stresses":     op2.solid_stresses(),
    "bar_stresses":       op2.bar_stresses(),
    "element_forces":     op2.element_forces(),
    "spc_forces":         op2.spc_forces(),
    "applied_loads":      op2.applied_loads(),
    "strains":            op2.strains(),
    "eigenvalues":        op2.eigenvalues(),
}

rows = []
for name, d in all_results.items():
    if d:
        for sc, df in d.items():
            rows.append({"result": name, "subcase": sc, "rows": len(df), "cols": len(df.columns)})
    else:
        rows.append({"result": name, "subcase": "-", "rows": 0, "cols": 0})

# Grid Point Weight Generator (OGPWG) — single result, not subcase-keyed
gw = op2.grid_weight()
if gw:
    rows.append({"result": "grid_weight", "subcase": "-", "rows": 1, "cols": len(gw["summary"].columns)})
else:
    rows.append({"result": "grid_weight", "subcase": "-", "rows": 0, "cols": 0})

pd.DataFrame(rows)

In [ ]:
## 10 — Grid Point Weight (OGPWG)
# mass, centre of gravity, and inertia matrix from the OGPWG table.

In [ ]:
gw = op2.grid_weight()
if gw:
    print(f"Total mass  : {gw['mass']:.6g}")
    print(f"CG (X, Y, Z): {[round(v, 6) for v in gw['cg']]}")
    display(gw["summary"])
else:
    print("No OGPWG table found in this file.")

In [ ]:
## 11 — Corner Stresses (OES1X1 with corner nodes)
# 5 rows per CQUAD4: GRID=0 is centroid, GRID!=0 are the 4 corner node IDs.
sc_corners = op2.stresses_with_corners()
for sc, df in sc_corners.items():
    centroids = df[df["GRID"] == 0]
    corners   = df[df["GRID"] != 0]
    print(f"Subcase {sc}: {len(df)} rows  ({len(centroids)} centroids + {len(corners)} corner rows)")
    display(df.head(10))

In [ ]:
## 12 — find_by_eid  &  to_csv

# -- 12a: look up all results for a single element --
EID_QUERY = 1
hits = op2.find_by_eid(EID_QUERY)
print(f"Results found for EID={EID_QUERY}:")
for result_name, df in hits.items():
    print(f"  {result_name}: {len(df)} row(s)")
    display(df)

# -- 12b: export every result table to CSV --
import tempfile, os
out_dir = r"c:\Users\evanc\Dropbox\Coding\Python\OP2_reader\csv_export"
written = op2.to_csv(out_dir)
print(f"\nCSV files written to: {out_dir}")
for fname, fpath in sorted(written.items()):
    size_kb = os.path.getsize(fpath) / 1024
    print(f"  {fname:<35s}  {size_kb:7.1f} kB")

In [ ]:
# §13 — Performance: cache speedup
import time, importlib, op2_native.reader as _r
importlib.reload(_r)
from op2_native.reader import OP2

fresh = OP2(OP2_FILE)

# First call — cold (decode from bytes)
t0 = time.perf_counter()
_ = fresh.stresses()
cold_ms = (time.perf_counter() - t0) * 1000

# Second call — warm (from cache)
t0 = time.perf_counter()
_ = fresh.stresses()
warm_ms = (time.perf_counter() - t0) * 1000

print(f"stresses()  cold: {cold_ms:.2f} ms")
print(f"stresses()  warm: {warm_ms:.2f} ms  ({cold_ms/max(warm_ms,0.001):.0f}× speedup)")

# Confirm cache is populated
print(f"\nCache keys: {list(fresh._cache.keys())}")
fresh.clear_cache()
print(f"After clear_cache(): {list(fresh._cache.keys())}")

In [ ]:
# §14 — find_by_grid(grid_id): nodal results lookup
GRID_QUERY = 51   # pick any node ID from the model

grid_hits = op2.find_by_grid(GRID_QUERY)
print(f"find_by_grid({GRID_QUERY}) found results in: {list(grid_hits.keys())}\n")
for name, df in grid_hits.items():
    print(f"--- {name} ({len(df)} rows) ---")
    display(df.head(6))

In [ ]:
## 15 — OP2 repr  &  Results object

# The OP2 repr now shows a summary of all non-empty result types
print(repr(op2))
print()

# Results gives flat attribute access to all tables for one subcase
from op2_native import Results
r = op2.results(subcase=1)
print(repr(r))
print()

# Access individual tables directly as attributes
print("Stresses (first 3 rows):")
display(r.stresses.head(3))

print("Element forces (first 3 rows):")
display(r.element_forces.head(3))

In [ ]:
## 16 — Plotly Visualisations
from op2_native import plots

r = op2.results(subcase=1)

# Von Mises stress — max of both fibers, coloured bar chart sorted by EID
plots.plot_vm_stress(r.stresses, fiber="max")

In [ ]:
# Displacement magnitude scatter — each node coloured by |U|
plots.plot_displacement_magnitude(r.displacements, component="mag")

In [ ]:
# Element force NX (membrane x) — diverging red-blue colorscale
plots.plot_element_forces(r.element_forces, component="NX")

In [ ]:
# VM1 stress histogram with mean line
plots.plot_stress_histogram(r.stresses, column="VM1", bins=40)

## §17 — envelope(), describe(), and new plots

In [ ]:
# Reload modules so changes to reader.py and plots.py are picked up
import importlib, op2_native, op2_native.reader, op2_native.plots
importlib.reload(op2_native.plots)
importlib.reload(op2_native.reader)
importlib.reload(op2_native)

from op2_native import OP2
op2 = OP2(OP2_FILE)
r   = op2.results(1)
print("Modules reloaded OK")

In [ ]:
# envelope() — worst-case VM1 stress across all subcases
env = op2.envelope("stresses", "VM1", "absmax")
print(f"Envelope shape: {env.shape}")
print(f"Governing column range: {env['VM1'].min():.4g} → {env['VM1'].max():.4g}")
display(env.head(10))

In [ ]:
# describe() — statistical summary of all non-empty result tables
desc = op2.describe()
print(f"describe() shape: {desc.shape}")
display(desc.round(4))

In [ ]:
# Top-20 stressed elements — horizontal bar chart with EID labels
op2_native.plots.plot_top_n_stress(r.stresses, n=20, fiber="max")

In [ ]:
# Principal stress comparison: MAJOR vs MINOR vs VM (fiber 1, top 50 elements by VM)
op2_native.plots.plot_principal_stress(r.stresses, fiber="1", max_elements=50)

In [ ]:
# Polar scatter of in-plane element forces: NX at 0°, NY at 90°, NXY at 45°
op2_native.plots.plot_forces_polar(r.element_forces)

## §18 — subcases(), eqexin(), plot_stress_summary()

In [ ]:
import importlib, op2_native, op2_native.reader, op2_native.plots
importlib.reload(op2_native.plots)
importlib.reload(op2_native.reader)
importlib.reload(op2_native)
from op2_native import OP2
op2 = OP2(OP2_FILE)
r   = op2.results(1)
print("Reloaded")

In [ ]:
# subcases() — cross-tab of result type vs subcase ID
sc_table = op2.subcases()
print(sc_table.to_string())
display(sc_table)

In [ ]:
# eqexin() — GRID to DOF-pointer mapping
eq = op2.eqexin()
print(f"EQEXIN: {len(eq)} rows, GRID range {eq['GRID'].min()}–{eq['GRID'].max()}")
print(f"Constrained grids (EQTYPE==0): {(eq['EQTYPE']==0).sum()}")
display(eq.head(10))

In [ ]:
# Stress component summary — 4×2 histogram grid for all components
op2_native.plots.plot_stress_summary(r.stresses, fiber="1")

## §19 — Multi-record data fix verification

**Problem (fixed 2026-03-30):** All three data decoders (`decode_oes1x1_shell`, `decode_oes1x1_shell_corners`, `decode_oef1`) only read the **first** Fortran data record per table, silently dropping the remaining records. For a ~3,661-element model the OES1X1 table has 5 data records and OEF1X has 3, so ~80% of elements were missing from all outputs.

**Fix:** `collect_data_records_after()` + `load_data_bytes()` added to `oes_peek.py`. Both shell decoders and the force decoder now concatenate all contiguous data records before decoding.

In [ ]:
# §19a — Verify full element counts after multi-record fix
import sys
for mod in list(sys.modules.keys()):
    if "op2_native" in mod:
        del sys.modules[mod]

from op2_native import OP2
op2 = OP2(OP2_FILE)
r = op2.results(1)

st = r.stresses
sc = r.stresses_corners
ef = r.element_forces

print(f"stresses()        rows: {len(st):>6,}   unique EIDs: {st['EID'].nunique():>5,}")
print(f"stresses_corners  rows: {len(sc):>6,}   unique EIDs: {sc['EID'].nunique():>5,}")
print(f"element_forces()  rows: {len(ef):>6,}   unique EIDs: {ef['EID'].nunique():>5,}")
print()
print(f"stresses   attrs: {st.attrs.get('all_data_records')}")
print(f"forces     attrs: {ef.attrs.get('all_data_records')}")


## §19b — Next steps

- **`oes_bar.py`** — `decode_oes_bar` still uses `first_stress_record_after` + single `rec.data`. Update to `load_data_bytes` (same pattern, but bar models are often small so may not be urgent).
- **`lama.py`** — `_first_data_record_after` is its own local helper; verify that eigenvalue tables never span multiple records (they typically don't for modal analyses).
- **Solid elements** — no solid-element OES decoder exists yet; if needed, add `oes1x1_solid.py` following the same `load_data_bytes` pattern.
- **OUG / grid-point results** — already single-record in typical models; no change needed unless very large grids are encountered.
- **Test with pyNastran** — cross-check EID sets and stress values against `pyNastran.op2.OP2` as a reference implementation.

In [1]:
EIDS_TO_CHECK = [1, 2, 3] # CBEAM element IDs
GRIDS_TO_CHECK = [1, 2, 3] # node IDs

import sys, warnings

from matplotlib.pyplot import title
warnings.filterwarnings("ignore")
for mod in list(sys.modules.keys()):
    if "op2_native" in mod:
        del sys.modules[mod]

from op2_native import OP2
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:.6e}".format)

CYL_OP2 = r"run_files/model/cylinder-0000.op2"
cyl = OP2(CYL_OP2)
print("Loaded:", CYL_OP2)

all_sc = sorted(
set(cyl.displacements()) | set(cyl.bar_stresses()) |
set(cyl.element_forces()) | set(cyl.bush_stresses()) |
set(cyl.bush_forces()) | set(cyl.solid_stresses())
)
print(f"Subcases found: {all_sc}\n")

def section(title):
    print(f"\n{'═'*72}\n {title}\n{'═'*72}")

for sc in all_sc:
    print(f"\n\n{'#'*72}")
    print(f" SUBCASE {sc}")
    print(f"{'#'*72}")

    # ── Displacements ─────────────────────────────────────────────────────────
    disp = cyl.displacements().get(sc)
    if disp is not None:
        d = disp[disp["GRID"].isin(GRIDS_TO_CHECK)]
        if len(d):
            section("DISPLACEMENTS  (T1/T2/T3 = translations,  R1/R2/R3 = rotations)")
            display(d.set_index("GRID").drop(columns=["CP"], errors="ignore"))

    # ── CBEAM stresses ────────────────────────────────────────────────────────
    bstr = cyl.bar_stresses().get(sc)
    if bstr is not None:
        b = bstr[bstr["EID"].isin(EIDS_TO_CHECK)]
        if len(b):
            section("CBEAM STRESSES  (SXC/SXD/SXE/SXF at 4 fiber corners,  SMAX/SMIN)")
            display(b.set_index(["EID", "GRID"]))

    # ── CBEAM forces ──────────────────────────────────────────────────────────
    ef = cyl.element_forces().get(sc)
    if ef is not None:
        e = ef[ef["EID"].isin(EIDS_TO_CHECK)]
        if len(e):
            section("CBEAM FORCES  (BM1/BM2 = bending moments,  WS1/WS2 = web shears,  AF = axial,  TRQ = torque)")
            display(e.set_index(["EID", "GRID"]))

    # ── CBUSH stresses ────────────────────────────────────────────────────────
    bush_s = cyl.bush_stresses().get(sc)
    if bush_s is not None:
        bs = bush_s[bush_s["EID"].isin(EIDS_TO_CHECK)]
        if len(bs):
            section("CBUSH STRESSES  (EX/EY/EZ = axial/shear,  ETX/ETY/ETZ = rotational)")
            display(bs.set_index("EID"))

    # ── CBUSH forces ──────────────────────────────────────────────────────────
    bush_f = cyl.bush_forces().get(sc)
    if bush_f is not None:
        bf = bush_f[bush_f["EID"].isin(EIDS_TO_CHECK)]
        if len(bf):
            section("CBUSH FORCES  (FX/FY/FZ = forces,  MX/MY/MZ = moments)")
            display(bf.set_index("EID"))

    # ── CTETRA stresses ───────────────────────────────────────────────────────
    sol = cyl.solid_stresses().get(sc)
    if sol is not None:
        ss = sol[sol["EID"].isin(EIDS_TO_CHECK)]
        if len(ss):
            section("CTETRA STRESSES  (GRID=0 → centroid,  GRID!=0 → corner node)")
            display(ss.set_index(["EID", "GRID"]))

    print("\n\nDone. Adjust EIDS_TO_CHECK / GRIDS_TO_CHECK at the top and re-run the two cells.")

Loaded: run_files/model/cylinder-0000.op2
Subcases found: [1, 2]



########################################################################
 SUBCASE 1
########################################################################

════════════════════════════════════════════════════════════════════════
 DISPLACEMENTS  (T1/T2/T3 = translations,  R1/R2/R3 = rotations)
════════════════════════════════════════════════════════════════════════


,DX,DY,DZ,RX,RY,RZ
GRID,,,,,,
1,4.075506e-09,-9.748280e-09,-1.017677e-06,-3.106600e-09,-1.715314e-09,1.533204e-11
2,4.069524e-09,-9.748869e-09,-1.018955e-06,-3.106600e-09,-1.715314e-09,1.533204e-11
3,4.063772e-09,-9.750615e-09,-1.020316e-06,-3.106600e-09,-1.715314e-09,1.533204e-11



════════════════════════════════════════════════════════════════════════
 CBEAM STRESSES  (SXC/SXD/SXE/SXF at 4 fiber corners,  SMAX/SMIN)
════════════════════════════════════════════════════════════════════════


SD          SXC          SXD          SXE          SXF         SMAX         SMIN         MS_T         MS_C
EID GRID                                                                                                                     
1   7573 0.000000e+00 2.554451e+00 3.109889e+00 2.554987e+00 1.999550e+00 3.109889e+00 1.999550e+00 1.401298e-45 1.401298e-45
    225  1.000000e+00 2.562163e+00 1.994527e+00 2.547276e+00 3.114912e+00 3.114912e+00 1.994527e+00 1.401298e-45 1.401298e-45
2   7573 0.000000e+00 2.337392e+00 2.045326e+00 2.774824e+00 3.066890e+00 3.066890e+00 2.045326e+00 1.401298e-45 1.401298e-45
    226  1.000000e+00 2.767710e+00 3.071831e+00 2.344507e+00 2.040386e+00 3.071831e+00 2.040386e+00 1.401298e-45 1.401298e-45
3   7573 0.000000e+00 2.157241e+00 2.171009e+00 2.957813e+00 2.944045e+00 2.957813e+00 2.157241e+00 1.401298e-45 1.401298e-45
    227  1.000000e+00 2.950626e+00 2.948881e+00 2.164427e+00 2.166173e+00 2.950626e+00 2.164427e+00 1.401298e-45 1.401298e-45


════════════════════════════════════════════════════════════════════════
 CBEAM FORCES  (BM1/BM2 = bending moments,  WS1/WS2 = web shears,  AF = axial,  TRQ = torque)
════════════════════════════════════════════════════════════════════════


...



Done. Adjust EIDS_TO_CHECK / GRIDS_TO_CHECK at the top and re-run the two cells.


########################################################################
 SUBCASE 2
########################################################################

════════════════════════════════════════════════════════════════════════
 CBEAM STRESSES  (SXC/SXD/SXE/SXF at 4 fiber corners,  SMAX/SMIN)
════════════════════════════════════════════════════════════════════════


SD           SXC          SXD           SXE          SXF         SMAX          SMIN         MS_T         MS_C
EID GRID                                                                                                                        
1   7573 0.000000e+00 -8.946110e+00 1.616876e-02  8.974891e+00 1.261272e-02 8.974891e+00 -8.946110e+00 1.401298e-45 1.401298e-45
    225  1.000000e+00  8.580526e+00 1.382561e-02 -8.551744e+00 1.495587e-02 8.580526e+00 -8.551744e+00 1.401298e-45 1.401298e-45
2   7573 0.000000e+00  9.823775e+00 7.214805e-01 -8.025699e+00 1.076596e+00 9.823775e+00 -8.025699e+00 1.401298e-45 1.401298e-45
    226  1.000000e+00 -7.629358e+00 1.076315e+00  9.427435e+00 7.217621e-01 9.427435e+00 -7.629358e+00 1.401298e-45 1.401298e-45
3   7573 0.000000e+00  1.049133e+01 1.402433e+00 -7.193587e+00 1.895307e+00 1.049133e+01 -7.193587e+00 1.401298e-45 1.401298e-45
    227  1.000000e+00 -6.792391e+00 1.895805e+00  1.009013e+01 1.401935e+00 1.009013e+01 -6.792391e+00 1.401298e-45 1.401298e-45


════════════════════════════════════════════════════════════════════════
 CBEAM FORCES  (BM1/BM2 = bending moments,  WS1/WS2 = web shears,  AF = axial,  TRQ = torque)
════════════════════════════════════════════════════════════════════════


SD           BM1           BM2           WS1           WS2           AF          TRQ
EID GRID                                                                                               
1   7573 0.000000e+00  4.504039e-01 -8.937322e-05  4.273404e-01 -5.713154e-05 7.233575e-03 2.052412e-03
    225  1.000000e+00 -4.305809e-01  2.840646e-05  4.273404e-01 -5.713154e-05 7.233575e-03 2.052412e-03
2   7573 0.000000e+00 -4.486062e-01  8.925039e-03 -4.255483e-01  8.651694e-03 4.519061e-01 1.911875e-03
    226  1.000000e+00  4.286840e-01 -8.910886e-03 -4.255483e-01  8.651694e-03 4.519061e-01 1.911875e-03
3   7573 0.000000e+00 -4.444704e-01  1.238727e-02 -4.214175e-01  1.202955e-02 8.288125e-01 1.507403e-03
    227  1.000000e+00  4.243041e-01 -1.241229e-02 -4.214175e-01  1.202955e-02 8.288125e-01 1.507403e-03



Done. Adjust EIDS_TO_CHECK / GRIDS_TO_CHECK at the top and re-run the two cells.
